# Task 4
This serves as a template which will guide you through the implementation of this task. It is advised to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [32]:
import os
# We disable low-level log outputs by default to keep the terminal clean
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
# Add any other imports you need here
from transformers import AutoModel
from transformers import AutoTokenizer
from transformers import get_linear_schedule_with_warmup



Depending on your approach, you might need to adapt the structure of this template or parts not marked by TODOs.
It is not necessary to completely follow this template. Feel free to add more code and delete any parts that are not required.

In [33]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32  # TODO: Set the batch size according to both training performance and available memory
NUM_EPOCHS = 5  # TODO: Set the number of epochs

train_val = pd.read_csv("train.csv")
test_val = pd.read_csv("test_no_score.csv")

In [34]:
# TODO: Fill out SentimentDataset
class SentimentDataset(Dataset):
    def __init__(self, texts, tokenizer, labels=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        # TODO: Tokenize and return a {input_ids, attention_mask, labels} dictionary, or just {input_ids, attention_mask} for test data.
        encoding = self.tokenizer(
            self.texts[index],
            padding='max_length',
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )

        item = {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze()
        }

        # add labels if available (training)
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)

        return item

In [35]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
#train_dataset = SentimentDataset(train_val["text"].tolist(), train_val["label"].tolist())
#test_dataset = SentimentDataset(test_val["text"].tolist())
train_dataset = SentimentDataset(
    train_val["sentence"].tolist(),
    tokenizer=tokenizer,
    labels=train_val["score"].tolist()
)
test_dataset = SentimentDataset(
    test_val["sentence"].tolist(),
    tokenizer=tokenizer
)

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True,
                          num_workers=0,         # If you want to utilize multi-processing, set this to the number of your available cores!
                          pin_memory=True)
test_loader = DataLoader(dataset=test_dataset,
                         batch_size=BATCH_SIZE,
                         shuffle=False,
                         num_workers=0,          # If you want to utilize multi-processing, set this to the number of your available cores!
                         pin_memory=True)
# Additional code if needed

In [36]:
# TODO: Fill out SentimentClassifier
class SentimentClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("distilbert-base-uncased")
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits = self.fc(cls_output)
        return logits

model = SentimentClassifier().to(DEVICE)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
# TODO: Setup loss function, optimizer, and scheduler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

model.train()
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, total=len(train_loader)):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        loss = criterion(outputs, batch["labels"])

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}, Loss: {avg_loss:.4f}")


        # TODO: Set up training loop

100%|██████████| 391/391 [02:22<00:00,  2.74it/s]


Epoch 1/5, Loss: 0.3487


100%|██████████| 391/391 [02:21<00:00,  2.77it/s]


Epoch 2/5, Loss: 0.1976


100%|██████████| 391/391 [02:21<00:00,  2.77it/s]


Epoch 3/5, Loss: 0.1021


100%|██████████| 391/391 [02:21<00:00,  2.77it/s]


Epoch 4/5, Loss: 0.0507


100%|██████████| 391/391 [02:21<00:00,  2.77it/s]

Epoch 5/5, Loss: 0.0292


In [38]:
model.eval()
with torch.no_grad():
    results = []
    for batch in tqdm(test_loader, total=len(test_loader)):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        # TODO: Set up evaluation loop
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        predictions = torch.argmax(outputs, dim=1)

        results.append(predictions.cpu().numpy())

    with open("result.txt", "w") as f:
        for val in np.concatenate(results):
            f.write(f"{val}\n")

100%|██████████| 32/32 [00:04<00:00,  7.98it/s]
